In [1]:
!pip install vaderSentiment

  Using cached vaderSentiment-3.3.2-py2.py3-none-any.whl.metadata (572 bytes)
Using cached vaderSentiment-3.3.2-py2.py3-none-any.whl (125 kB)


In [2]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [3]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment_score(text):
    return analyzer.polarity_scores(text)["compound"]

In [8]:
get_sentiment_score("This movie was amazing and feel good")

0.7717

In [9]:
def rerank_with_sentiment(recs):
    recs = recs.copy()

    # temporary default audience sentiment score
    recs["sentiment_score"] = 0.7

    recs["final_score_sentiment"] = (
        0.8 * recs["final_score"] +
        0.2 * recs["sentiment_score"]
    )

    return recs.sort_values("final_score_sentiment", ascending=False)

In [10]:
def explain_recommendation(row, liked_movie_title=None):
    reasons = []

    if row["svd_score"] > 0.75:
        reasons.append("the SVD model predicts you may rate it highly")

    if row["content_score"] > 0.5 and liked_movie_title:
        reasons.append(f"it is similar to {liked_movie_title}")

    if row["collab_score"] > 0.7:
        reasons.append("users generally rated this movie well")

    reasons.append(f"it belongs to genres like {row['genres']}")

    return "Recommended because " + ", ".join(reasons) + "."

In [13]:
def hybrid_recommend(user_id, liked_movie_title=None, top_n=10):
    all_movie_ids = movies_small["movieId"].unique()

    watched_movies = ratings_small[
        ratings_small["userId"] == user_id
    ]["movieId"].unique()

    unwatched_movies = [
        movie_id for movie_id in all_movie_ids
        if movie_id not in watched_movies
    ]

    results = []

    for movie_id in unwatched_movies:
        pred = svd_model.predict(user_id, movie_id)
        svd_score = pred.est / 5.0

        avg_rating = ratings_small[
            ratings_small["movieId"] == movie_id
        ]["rating"].mean()

        if pd.isna(avg_rating):
            collab_score = 0.5
        else:
            collab_score = avg_rating / 5.0

        content_score = 0.5

        if liked_movie_title and liked_movie_title in indices:
            liked_idx = indices[liked_movie_title]

            movie_row = movies_small[movies_small["movieId"] == movie_id]

            if not movie_row.empty:
                movie_idx = movie_row.index[0]
                content_score = cosine_sim[liked_idx][movie_idx]

        final_score = (
            0.4 * svd_score +
            0.3 * collab_score +
            0.3 * content_score
        )

        results.append((movie_id, svd_score, collab_score, content_score, final_score))

    result_df = pd.DataFrame(
        results,
        columns=["movieId", "svd_score", "collab_score", "content_score", "final_score"]
    )

    result_df = result_df.merge(movies_small, on="movieId")

    result_df = result_df.sort_values("final_score", ascending=False)

    return result_df.head(top_n)[
        ["movieId", "title", "genres", "svd_score", "collab_score", "content_score", "final_score"]
    ]

In [15]:
ratings = pd.read_csv("../data/raw/ratings.csv")
movies = pd.read_csv("../data/raw/movies.csv")

ratings_small = ratings.sample(200000, random_state=42)

movies["genres_clean"] = movies["genres"].str.replace("|", " ", regex=False)
movies_small = movies.head(5000).copy().reset_index(drop=True)

In [18]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split

In [19]:
reader = Reader(rating_scale=(0.5, 5.0))

data = Dataset.load_from_df(
    ratings_small[["userId", "movieId", "rating"]],
    reader
)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

svd_model = SVD()
svd_model.fit(trainset)

In [21]:
tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(movies_small["genres_clean"].fillna(""))

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

indices = pd.Series(
    movies_small.index,
    index=movies_small["title"]
).drop_duplicates()

In [22]:
recs = hybrid_recommend(
    user_id=1,
    liked_movie_title="Toy Story (1995)",
    top_n=10
)

recs = rerank_with_sentiment(recs)

recs["reason"] = recs.apply(
    lambda row: explain_recommendation(row, "Toy Story (1995)"),
    axis=1
)

recs[["title", "genres", "final_score_sentiment", "reason"]]

,title,genres,final_score_sentiment,reason
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,0.817365,Recommended because the SVD model predicts you...
3021,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.817018,Recommended because the SVD model predicts you...
4932,"Flight of Dragons, The (1982)",Adventure|Animation|Children|Drama|Fantasy,0.811692,Recommended because the SVD model predicts you...
4780,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,0.809430,Recommended because the SVD model predicts you...
705,Wallace & Gromit: The Best of Aardman Animatio...,Adventure|Animation|Comedy,0.805150,Recommended because the SVD model predicts you...
3912,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,0.802382,Recommended because it is similar to Toy Story...
729,Wallace & Gromit: A Close Shave (1995),Animation|Children|Comedy,0.799625,Recommended because the SVD model predicts you...
4201,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Ro...,0.786332,Recommended because it is similar to Toy Story...
1108,Monty Python and the Holy Grail (1975),Adventure|Comedy|Fantasy,0.782631,Recommended because the SVD model predicts you...
1944,"Black Cauldron, The (1985)",Adventure|Animation|Children|Fantasy,0.772873,Recommended because it is similar to Toy Story...
